# FINAL: Earth analogs for Mars (reference site: Viking 2, Utopia Planitia)
Earth data: NASA MERRA-2 (Earthdata). Mars reference values: Viking 2 data from the PDS Atmospheres Node.
Run the cells in order. Edit only section 2.

In [ ]:
!pip install -q earthaccess xarray netCDF4 pandas matplotlib folium

## 1) Login to Earthdata (also approve the GES DISC application in your Earthdata profile)

In [ ]:
import earthaccess
auth = earthaccess.login()

## 2) Settings (EDIT HERE)
- `SITES`: the team's Earth sites (lat, lon). Placeholders below, replace them.
- `MARS`: paste the dictionary printed by the Viking extraction cell.

In [ ]:
YEAR = 2022

SITES = {
    "Lut Desert (Iran)":        (30.5, 59.0),
    "Gilf Kebir (Egypt)":       (23.3, 26.1),
    "Tibesti (Chad)":           (21.0, 17.5),
    "Puna de Atacama (Chile)":  (-25.0, -67.0),
    "Taklamakan (China)":       (39.0, 83.0),
    "Dry Valleys (Antarctica)": (-77.5, 162.5),
}

# >>> PASTE the MARS dictionary from the Viking cell here (these are only fallback values) <<<
MARS = {
    "ts_mean_K": 205.0,
    "elevation_m": -4000,
    "dust_aod": 0.5,
    "lai": 0.0,
    "soil_wetness": 0.0,
    "wind_ms": 4.5,
    "pressure_Pa": 890,
}

SCALE   = {"ts_mean_K": 50, "elevation_m": 5000, "dust_aod": 0.5, "lai": 1.0, "soil_wetness": 1.0, "wind_ms": 5.0}
WEIGHTS = {"ts_mean_K": 1, "elevation_m": 0, "dust_aod": 1, "lai": 1, "soil_wetness": 1, "wind_ms": 1}
# elevation weight is 0: Utopia is ~4 km below the Mars datum, so almost every Earth site scores 0 and it cannot discriminate

## 3) Download MERRA-2 (monthly files)

In [ ]:
import xarray as xr

def get_files(short_name, **kw):
    res = earthaccess.search_data(short_name=short_name, **kw)
    print(short_name, "->", len(res), "granules")
    return earthaccess.download(res, "data")

period = dict(temporal=(f"{YEAR}-01-01", f"{YEAR}-12-31"))
slv  = xr.open_mfdataset(get_files("M2TMNXSLV", **period), combine="by_coords")
aer  = xr.open_mfdataset(get_files("M2TMNXAER", **period), combine="by_coords")
lnd  = xr.open_mfdataset(get_files("M2TMNXLND", **period), combine="by_coords")
cnst = xr.open_dataset(get_files("M2C0NXASM")[0])

# Viking measured AIR temperature and wind at ~1.6 m, so use 2 m variables when they exist
TVAR = "T2M" if "T2M" in slv.data_vars else "TS"
UVAR, VVAR = ("U2M", "V2M") if "U2M" in slv.data_vars and "V2M" in slv.data_vars else ("U10M", "V10M")
print("temperature variable:", TVAR, "| wind variables:", UVAR, VVAR)
for name, ds, names in [("slv", slv, [TVAR, UVAR, VVAR, "PS"]), ("aer", aer, ["DUEXTTAU"]),
                        ("lnd", lnd, ["LAI", "GWETTOP"]), ("cnst", cnst, ["PHIS"])]:
    miss = [v for v in names if v not in ds.data_vars]
    print(name, "OK" if not miss else f"MISSING {miss} | available: {list(ds.data_vars)}")

## 4) Values for each site

In [ ]:
import numpy as np
import pandas as pd

def at(ds, var, lat, lon):
    return ds[var].sel(lat=lat, lon=lon, method="nearest")

rows = []
for name, (lat, lon) in SITES.items():
    t   = at(slv, TVAR, lat, lon).values
    u   = at(slv, UVAR, lat, lon).values
    v   = at(slv, VVAR, lat, lon).values
    ps  = at(slv, "PS", lat, lon).values
    d   = at(aer, "DUEXTTAU", lat, lon).values
    lai = at(lnd, "LAI", lat, lon).values
    wet = at(lnd, "GWETTOP", lat, lon).values
    phi = at(cnst, "PHIS", lat, lon).values.squeeze()
    rows.append({"site": name, "lat": lat, "lon": lon,
        "ts_mean_K": float(np.nanmean(t)), "ts_seasonal_range_K": float(np.nanmax(t) - np.nanmin(t)),
        "elevation_m": float(phi) / 9.80665, "dust_aod": float(np.nanmean(d)),
        "lai": float(np.nanmean(lai)), "soil_wetness": float(np.nanmean(wet)),
        "wind_ms": float(np.nanmean(np.sqrt(u**2 + v**2))), "pressure_Pa": float(np.nanmean(ps))})
df = pd.DataFrame(rows)
df["pressure_ratio_to_Mars"] = df["pressure_Pa"] / MARS["pressure_Pa"]
df.round(3)

## 5) Similarity scores (missing variables are ignored, not counted as zero)

In [ ]:
for k in SCALE:
    df["score_" + k] = (100 * (1 - (df[k] - MARS[k]).abs() / SCALE[k])).clip(lower=0)

def total_score(frame, weights):
    cols = ["score_" + k for k in SCALE]
    W = pd.Series({"score_" + k: weights[k] for k in SCALE}, dtype=float)
    ok = frame[cols].notna()
    return (frame[cols].fillna(0) * W).sum(axis=1) / (ok * W).sum(axis=1)

df["score_total"] = total_score(df, WEIGHTS)
df = df.sort_values("score_total", ascending=False).reset_index(drop=True)
df.round(1)

## 6) Pressure: where the analogy fails

In [ ]:
df[["site", "pressure_Pa", "pressure_ratio_to_Mars"]].round(1)

## 7) Chart

In [ ]:
import matplotlib.pyplot as plt
cols = ["score_" + k for k in SCALE if WEIGHTS[k] > 0]
ax = df.set_index("site")[cols].plot(kind="bar", figsize=(11, 5))
ax.set_ylabel("Similarity to Mars (0-100)"); ax.set_title("Similarity to the Viking 2 site, per variable")
plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.savefig("scores.png", dpi=150); plt.show()

## 8) Sensitivity analysis (does the ranking change with the weights?)

In [ ]:
scen = {
    "equal":          {"ts_mean_K": 1, "elevation_m": 0, "dust_aod": 1, "lai": 1, "soil_wetness": 1, "wind_ms": 1},
    "temperature x3": {"ts_mean_K": 3, "elevation_m": 0, "dust_aod": 1, "lai": 1, "soil_wetness": 1, "wind_ms": 1},
    "dust+wind x3":   {"ts_mean_K": 1, "elevation_m": 0, "dust_aod": 3, "lai": 1, "soil_wetness": 1, "wind_ms": 3},
    "with elevation": {"ts_mean_K": 1, "elevation_m": 1, "dust_aod": 1, "lai": 1, "soil_wetness": 1, "wind_ms": 1},
}
sens = pd.DataFrame({"site": df["site"]})
for label, w in scen.items():
    sens[label] = total_score(df, w)
    sens[label + " (rank)"] = sens[label].rank(ascending=False).astype(int)
sens.round(1)

## 9) Map

In [ ]:
import folium
m = folium.Map(location=[20, 30], zoom_start=2, tiles="CartoDB positron")
for _, r in df.iterrows():
    tip = (f"<b>{r['site']}</b><br>Total: {r['score_total']:.0f}<br>Temp {r['score_ts_mean_K']:.0f} | Dust {r['score_dust_aod']:.0f}"
           f"<br>Vegetation {r['score_lai']:.0f} | Soil moisture {r['score_soil_wetness']:.0f} | Wind {r['score_wind_ms']:.0f}"
           f"<br>Pressure: {r['pressure_ratio_to_Mars']:.0f}x Mars")
    color = "green" if r["score_total"] >= 70 else "orange" if r["score_total"] >= 40 else "red"
    folium.Marker([r["lat"], r["lon"]], popup=folium.Popup(tip, max_width=260), icon=folium.Icon(color=color)).add_to(m)
m.save("map.html")
m

## 10) Save everything (download, then upload to GitHub)

In [ ]:
df.to_csv("results.csv", index=False); sens.to_csv("sensitivity.csv", index=False)
pd.Series(MARS).to_csv("mars_reference_values.csv", header=["value"])
from google.colab import files
for f in ["results.csv", "sensitivity.csv", "mars_reference_values.csv", "map.html", "scores.png"]:
    files.download(f)